# Experimento pareado: LSTM32 de una capa frente a dos capas para F2 — Colab CPU

Se comparan dos entrenamientos desde cero con BCE normal: la LSTM32 causal actual de una capa frente a una LSTM32 apilada de dos capas. Solo cambia la profundidad recurrente. Se mantienen F2, dropout 0,2 en la cabeza de fusión, datos, semilla 42, optimizador, tasa, ventana y runtime CPU. El test 2025 permanece reservado.


## 1. Subir y verificar el paquete CPU

El paquete está en `runs/lstm_colab_lstm2layer_f2_cpu_v2.zip`. Para regenerarlo: `python3.13 scripts/empaquetar_lstm_depth.py --reference-config configs/experimento_lstm32_f2_lstm32x1_cpu_v1.json --candidate-config configs/experimento_lstm32_f2_lstm32x2_cpu_v1.json --output runs/lstm_colab_lstm2layer_f2_cpu_v2.zip`. Sube ese ZIP al panel **Archivos** de Colab. El notebook fija el SHA-256 esperado.


In [ ]:
from pathlib import Path
import hashlib, json, stat, subprocess, sys, zipfile
from google.colab import files

ROOT = Path('/content/f1-lstm-depth-cpu-v1')
PACKAGE = Path('/content/lstm_colab_lstm2layer_f2_cpu_v2.zip')
EXPECTED_PACKAGE_SHA256 = '9ef2e5ca476b5e51edaef1e7245c02a4905bf30489dccda738bc1c260aaca6f8'
if ROOT.exists(): raise FileExistsError('Usa una sesión limpia: /content/f1-lstm-depth-cpu-v1 ya existe.')
if not PACKAGE.is_file():
    uploaded = files.upload()
    if len(uploaded) != 1: raise ValueError('Sube exactamente el ZIP CPU de la comparación de profundidad LSTM.')
    name, content = next(iter(uploaded.items()))
    if Path(name).name != 'lstm_colab_lstm2layer_f2_cpu_v2.zip': raise ValueError('Se requiere lstm_colab_lstm2layer_f2_cpu_v2.zip; no uses el ZIP GPU ni otro archivo.')
    PACKAGE.write_bytes(content)
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()
package_hash = sha256(PACKAGE)
print('SHA-256 del paquete:', package_hash)
if package_hash != EXPECTED_PACKAGE_SHA256: raise ValueError('El hash no coincide con el ZIP CPU corregido.')
with zipfile.ZipFile(PACKAGE) as z:
    members=z.infolist(); names=set()
    for m in members:
        p=Path(m.filename)
        if p.is_absolute() or '..' in p.parts or '\\' in m.filename or stat.S_ISLNK(m.external_attr >> 16): raise ValueError(f'Ruta ZIP no segura: {m.filename}')
        if m.filename in names: raise ValueError(f'Miembro duplicado: {m.filename}')
        names.add(m.filename)
    manifest=json.loads(z.read('colab_package_manifest.json'))
    if manifest.get('artifact_version') != 'lstm_colab_lstm_depth_cpu_inputs_v1' or manifest.get('test_arrays_included') is not False: raise ValueError('Contrato de paquete no admitido o contiene test.')
    expected=set(manifest['files_sha256']) | {'colab_package_manifest.json'}
    if names != expected: raise ValueError('Los archivos del ZIP no coinciden con su manifiesto.')
    ROOT.mkdir(parents=True)
    z.extractall(ROOT)
for name, digest in manifest['files_sha256'].items():
    if sha256(ROOT/name) != digest: raise ValueError(f'SHA-256 incorrecto: {name}')
print('Manifiesto verificado:', len(manifest['files_sha256']), 'archivos; sin datos de prueba.')


## 2. Instalar dependencias y confirmar runtime CPU

Esta variante fija `device=cpu` en ambas configuraciones. No necesita GPU/CUDA; aunque Colab muestre una GPU disponible, el entrenamiento seguirá ejecutándose en CPU. La corrida puede tardar más que la pareja GPU.


In [ ]:
import subprocess, sys
from pathlib import Path
ROOT = Path('/content/f1-lstm-depth-cpu-v1')
if not (ROOT / 'requirements-colab-experiments.txt').is_file():
    raise RuntimeError('No encuentro el paquete CPU extraído en /content/f1-lstm-depth-cpu-v1. Ejecuta primero la sección 1 desde una sesión Colab activa.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT/'requirements-colab-experiments.txt')], check=True)
import torch
print('PyTorch:', torch.__version__, '| CUDA disponible:', torch.cuda.is_available(), '| dispositivo efectivo de ambas corridas: CPU')


## 3. Entrenar la pareja en CPU

La referencia tiene una LSTMCell y la candidata apila dos LSTMCell de 32 unidades. No se añade dropout entre capas: se conserva el dropout 0,2 existente en la cabeza de fusión para aislar únicamente la profundidad. Ambas usan BCE normal, selección F2 a umbral 0,5, early stopping y la misma cuadrícula F2 para el umbral congelado.


In [ ]:
import json, subprocess, sys
from pathlib import Path
from google.colab import files
ROOT=Path('/content/f1-lstm-depth-cpu-v1')
if not (ROOT/'scripts/entrenar_comparacion_lstm_depth.py').is_file(): raise RuntimeError('Extrae primero el paquete CPU.')
RUN=ROOT/'runs/lstm32_lstm2layer_f2_cpu_v1_seed42'
if RUN.exists() or RUN.with_suffix('.zip').exists(): raise FileExistsError('La salida ya existe; usa una sesión limpia.')
result=subprocess.run([sys.executable,'scripts/entrenar_comparacion_lstm_depth.py','--reference-config','configs/experimento_lstm32_f2_lstm32x1_cpu_v1.json','--candidate-config','configs/experimento_lstm32_f2_lstm32x2_cpu_v1.json','--output',str(RUN.relative_to(ROOT))],cwd=ROOT,check=False)
if result.returncode:
    status=RUN/'status.json'
    if status.is_file(): print(json.dumps(json.loads(status.read_text()),indent=2))
    raise RuntimeError(f'Falló la pareja; conserva artefactos. Código {result.returncode}.')
summary=json.loads((RUN/'pair_summary.json').read_text())
print(json.dumps(summary,indent=2))
files.download(str(RUN.with_suffix('.zip')))


## 4. Devolver resultados al proyecto

Guarda `lstm32_lstm2layer_f2_cpu_v1_seed42.zip` intacto en `runs/`. El ZIP contiene las dos corridas, sus pesos, historial, predicciones de validación, configuración, hashes y ambiente. La prueba 2025 no se utilizó.
